In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
import matplotlib.pyplot as plt
from cobra.sampling import OptGPSampler
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import os
from scipy.stats import linregress

In [2]:
cobra.Configuration().solver = "gurobi"

In [3]:
# model_path = './models/iMT1026v3.xml'
model_path = './models/iMT1026v3jup.xml'

model = cobra.io.read_sbml_model(model_path)

model

Set parameter Username
Academic license - for non-commercial use only - expires 2027-09-18


Name,iMT1026v3
Memory address,1aa9a490548
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [4]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x1aae8b2bdc8
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [5]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [6]:
#Tanquem l'entrada de Glicerol
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x1aae998e6c8
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [7]:
# Add qmeoh constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18)
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x1aae9999208
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [8]:
# Add qgluc constraints observed in chemostat cultivations from Sergi Monforte's doctoral thesis
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.72, -0.7)
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x1aae9970588
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.72
Upper bound,-0.7


In [9]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [10]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [11]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
ROL_exchange.bounds = (0,1000) #Value obtained from calculations of data from previous studies in our research group

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed from the model

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

## Reaction Ratios as constraints

### The experimental data was extracted from the paper:
#### "Metabolic flux analysis of recombinant Pichia pastoris growing on different glycerol/methanol mixtures by iterative fitting of NMR-derived 13C labelling data from proteinogenic amino acids" by Joel Jordà, 2014

In [12]:
ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

In [13]:
#WT Rol producer strain reference flux distribution
pfba_WT = cobra.flux_analysis.pfba(model)

In [14]:
pfba_WT.fluxes['Ex_rol']

0.0

In [15]:
#Obtain reference value for growth rate
RefGrowthRate = pfba_WT.fluxes['Ex_biomass']
RefGrowthRate

0.08372440263613617

## MOMA Simulations

### Carbon Source: Glc/MeOH

In [16]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

In [17]:
#Perform two different kind of simulations for candidates identification

#A MOMA simulation setting a minimum production of recombinant protein as the "mutant" condition
ROL_exchange.bounds = (0.003,1000)
moma_result = cobra.flux_analysis.moma(model, pfba_WT, 0)


#Set the recombinant protein production as the new objective function and constrain the growth rate to a 90% of the
#reference value, then carry out a pFBA simulation

ROL_exchange.bounds = (0, 1000)
Growth = model.reactions.get_by_id('Ex_biomass')
Growth.bounds = (RefGrowthRate*0.9, RefGrowthRate*0.9)
model.objective = "Ex_rol"
pfba_RolObj = cobra.flux_analysis.pfba(model)

In [18]:
WT_fluxes = pfba_WT.fluxes

# Calculate absolute flux changes relative to WT
moma_flux_changes = (
    moma_result.fluxes - WT_fluxes
).abs()

# Sort and obtain top 10
top10_moma = moma_flux_changes.sort_values(ascending=False).head(10)

print("TOP 10 REACTIONS - MOMA")
print("----------------------------------------")

for reaction_id, change in top10_moma.items():
    print(
        reaction_id,
        model.reactions.get_by_id(reaction_id).name,
        f"| Absolute flux change: {change:.6f}"
    )


# Calculate absolute flux changes relative to WT
pfba_flux_changes = (
    pfba_RolObj.fluxes - WT_fluxes
).abs()

# Sort and obtain top 10
top10_pfba = pfba_flux_changes.sort_values(ascending=False).head(10)

print("\nTOP 10 REACTIONS - pFBA ROL OBJECTIVE")
print("----------------------------------------")

for reaction_id, change in top10_pfba.items():
    print(
        reaction_id,
        model.reactions.get_by_id(reaction_id).name,
        f"| Absolute flux change: {change:.6f}"
    )

TOP 10 REACTIONS - MOMA
----------------------------------------
H2Ot H2O transport via diffusion | Absolute flux change: 0.012448
Ex_h2o H2O exchange | Absolute flux change: 0.012448
PPA inorganic diphosphatase | Absolute flux change: 0.005507
ADK1 adenylate kinase | Absolute flux change: 0.005041
ATPM ATP maintenance requirement | Absolute flux change: 0.004954
NTP1 ATPase | Absolute flux change: 0.004954
ATPH1 ATP diphosphohydrolase | Absolute flux change: 0.004868
biomass_glc_meoh_60/40 Biomass composition (g/g) - 60/40 Glucose/Methanol | Absolute flux change: 0.004847
COF Cofactor composition (mmol/g DCW) | Absolute flux change: 0.004847
growth Growth | Absolute flux change: 0.004847

TOP 10 REACTIONS - pFBA ROL OBJECTIVE
----------------------------------------
NADH2_u6cm NADH dehydrogenase, cytosolic/mitochondrial | Absolute flux change: 3.029123
MDH malate dehydrogenase | Absolute flux change: 2.667344
AKGMALtm alpha-ketoglutarate/malate transporter | Absolute flux change: 2.65

In [19]:
# Create Escher-compatible flux table
escher_fluxes = pd.DataFrame({
    "Reaction": pfba_RolObj.fluxes.index,
    "Flux": pfba_RolObj.fluxes.values
})

# Save as CSV
escher_fluxes.to_csv(
    "pFBA_ROL_objective_Escher.csv",
    index=False
)

print("CSV file saved as: pFBA_ROL_objective_Escher.csv")

# Preview
display(escher_fluxes.head())

CSV file saved as: pFBA_ROL_objective_Escher.csv


,Reaction,Flux
0,ADPtn,0.0
1,ATPtn,0.0
2,FACOAE140,0.0
3,FACOAL260,0.0
4,GLCter,0.0


In [20]:
# Create Escher-compatible flux table from WT pFBA
escher_fluxes_WT = pd.DataFrame({
    "Reaction": pfba_WT.fluxes.index,
    "Flux": pfba_WT.fluxes.values
})

# Remove reactions with zero flux
escher_fluxes_WT = escher_fluxes_WT[
    escher_fluxes_WT["Flux"].abs() > 1e-9
]

# Save as CSV
escher_fluxes_WT.to_csv(
    "pFBA_WT_Escher.csv",
    index=False
)

print(f"Saved {len(escher_fluxes_WT)} active reactions.")

# Preview
display(escher_fluxes_WT.head())

Saved 558 active reactions.


,Reaction,Flux
13,MIPCS324_SC,9.081439e-06
14,MIPCS326_SC,5.932212e-06
15,NADH2_u6mh,1.170570e+00
37,GLCCERS16d8,4.179638e-07
38,GLCCERS16m9,4.059092e-07
